In [35]:
import os
import json
import geojson
import requests

import pandas as pd
import geopandas as gpd
from geopandas.tools import geocode
from shapely.geometry import Point
from tqdm import tqdm

from osm2geojson import json2geojson

In [18]:
GTA_MUNICIPALITIES = [
    # City of Toronto (single-tier municipality)
    "Toronto",
    
    # Regional Municipality of Durham
    "Ajax",
    "Brock",
    "Clarington",
    "Oshawa",
    "Pickering",
    "Scugog",
    "Uxbridge",
    "Whitby",
    
    # Regional Municipality of Halton
    "Burlington",
    "Halton Hills",
    "Milton",
    "Oakville",
    
    # Regional Municipality of Peel
    "Brampton",
    "Caledon",
    "Mississauga",
    
    # Regional Municipality of York
    "Aurora",
    "East Gwillimbury",
    "Georgina",
    "King",
    "Markham",
    "Newmarket",
    "Richmond Hill",
    "Vaughan",
    "Whitchurch-Stouffville"
]

Prepare Business Axle data

In [19]:
# df_axle = pd.read_csv('../../data/venues/Canada DB 2024.csv')
# df_axle = df_axle[df_axle['STCITY'].isin(GTA_MUNICIPALITIES)]
# df_axle.to_csv('../../data/venues/business_axle_gta.csv', index=False)

df_axle = pd.read_csv('../../data/venues/business_axle_gta.csv')

df_axle = df_axle[[
    'CONAME', 'LOCNUM', 'SITE', 'STADDR', 
    'SUITE', 'STCITY', 'STATE', 'ZIP', 
    'ZIP4', 'ZIPP4F', 'DMCODE', 'STCODE', 
    'CNTYCD', 'CTRYCD',
    'NAICS', 'NAICSD', 
    'NAICS1', 'NAICS2', 'NAICS3', 'NAICS4',
]]

/tmp/ipykernel_427932/3681651342.py:5: DtypeWarning: Columns (15,29,54,74,80,85,89,90,94,95,110,119,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,186) have mixed types. Specify dtype option on import or set low_memory=False.
  df_axle = pd.read_csv('../../data/venues/business_axle_gta.csv')


In [23]:
codes = [
    '7111',  # Performing arts companies
    '7113',  # Promoters (presenters) of performing arts, sports and similar events
    '7114',  # Agents and managers for artists, athletes, entertainers and other public figures
    '7115',  # Independent artists, writers and performers
    '7121',  # Heritage institutions
    # '7139',  # Other amusement and recreation industries
]

# Filter rows where 'column_name' starts with any of the codes
df_naics = df_axle[df_axle['NAICS'].astype(str).str.startswith(tuple(codes))]
df_naics.to_csv('../../data/venues/axle_arts.csv', index=False)

In [49]:
df_naics_geo = pd.read_csv('../../data/venues/axle_arts_geo.csv').iloc[:, :22]
df_naics_geo = df_naics_geo[df_naics_geo['STCITY'] == 'Toronto']

In [50]:
# Create geometry from latitude and longitude
geometry = [Point(xy) for xy in zip(df_naics_geo['Geocodio Longitude'], df_naics_geo['Geocodio Latitude'])]

# Convert to GeoDataFrame
gdf_naics_geo = gpd.GeoDataFrame(df_naics_geo, geometry=geometry, crs="EPSG:4326")
gdf_naics_geo = gdf_naics_geo.drop(columns=['Geocodio Longitude', 'Geocodio Latitude'])

# Save to GeoPackage
OUT_PATH = "../../data/venues/axle_arts_geo.gpkg"
gdf_naics_geo.to_file(OUT_PATH, layer="naics", driver="GPKG")

# Check the first few rows
gdf_naics_geo.head()

,CONAME,LOCNUM,SITE,STADDR,SUITE,STCITY,STATE,ZIP,ZIP4,ZIPP4F,...,STCODE,CNTYCD,CTRYCD,NAICS,NAICSD,NAICS1,NAICS2,NAICS3,NAICS4,geometry
1,A & A Entertainment Network,12895041,NaN,124 Wolverleigh Blvd,NaN,Toronto,ON,M4J,1R9,M4J 1R9,...,35,2,CA,71119007,Other Performing Arts Companies,NaN,NaN,NaN,NaN,POINT (-79.32604 43.68481)
4,Alumnae Theatre,13050760,NaN,70 Berkeley St,NaN,Toronto,ON,M5A,2W6,M5A 2W6,...,35,2,CA,71111007,Theater Companies and Dinner Theaters,71111010.0,NaN,NaN,NaN,POINT (-79.36495 43.65274)
5,Anne Martin & Richard Howard,13079017,NaN,5 Woodbine Mews,NaN,Toronto,ON,M4L,3P1,M4L 3P1,...,35,2,CA,71151018,"Independent Artists, Writers, and Performers",NaN,NaN,NaN,NaN,POINT (-79.30499 43.66810)
6,Sro Management,13082086,NaN,189 Carlton St,NaN,Toronto,ON,M5A,2K7,M5A 2K7,...,35,2,CA,71119007,Other Performing Arts Companies,42512036.0,NaN,NaN,NaN,POINT (-79.37079 43.66337)
7,Arraymusic,13110622,NaN,155 Walnut Ave,NaN,Toronto,ON,M6J,3W3,M6J 3W3,...,35,2,CA,71113003,Musical Groups and Artists,71151015.0,NaN,NaN,NaN,POINT (-79.40991 43.64424)


Prepare OSM venue data

In [40]:
QUERY = """
[out:json][timeout:60];

// --- Define the area (Toronto) ---
area["name"="Toronto"]->.searchArea;
 
// --- Fetch cultural venues across amenity, tourism, leisure, building, and other keys ---
(
  // Amenity (performing arts & community)
  node["amenity"~"^(theatre|arts_centre|concert_hall|music_school|studio|cinema|library)$"](area.searchArea);
  way ["amenity"~"^(theatre|arts_centre|concert_hall|music_school|studio|cinema|library)$"](area.searchArea);
  relation["amenity"~"^(theatre|arts_centre|concert_hall|music_school|studio|cinema|library)$"](area.searchArea);
 
  // Tourism (galleries & museums)
  node["tourism"~"^(gallery|museum)$"](area.searchArea);
  way ["tourism"~"^(gallery|museum)$"](area.searchArea);
  relation["tourism"~"^(gallery|museum)$"](area.searchArea);
 
  // Leisure (auditorium, amphitheatre)
  node["leisure"~"^(auditorium|amphitheatre)$"](area.searchArea);
  way ["leisure"~"^(auditorium|amphitheatre)$"](area.searchArea);
  relation["leisure"~"^(auditorium|amphitheatre)$"](area.searchArea);
 
  // Building (cultural)
  node["building"="cultural"](area.searchArea);
  way ["building"="cultural"](area.searchArea);
  relation["building"="cultural"](area.searchArea);
 
  // Shops (ancillary spaces)
  node["shop"~"^(books|music|craft)$"](area.searchArea);
  way ["shop"~"^(books|music|craft)$"](area.searchArea);
  relation["shop"~"^(books|music|craft)$"](area.searchArea);
 
);
 
// --- Output full geometry for mapping ---
out body;
>;
out skel qt;
"""

OVERPASS_URL = 'https://overpass-api.de/api/interpreter'
RAW_PATH = "../../data/venues/eddit_venues.geojson"

In [41]:
response = requests.get(OVERPASS_URL, params={"data": QUERY})
response.raise_for_status()

result = json2geojson(response.json())

with open(RAW_PATH, "w") as f:
    geojson.dump(result, f)

In [42]:
gdf = gpd.read_file(RAW_PATH)

gdf = gdf.set_geometry("geometry")
gdf = gdf[gdf.geometry.notnull()]

# Preserve OSM bookkeeping fields before tag expansion
gdf = gdf.rename(
    columns={
        "type": "osm_element_type",
        "id": "osm_element_id",
    }
)

In [43]:
# --- CELL 5: Expand only a selected set of relevant tags ---
RELEVANT_TAGS = [
    "name",
    "name:en",
    "name:fr",
    "amenity",
    "tourism",
    "leisure",
    "building",
    "shop",
    "operator",
    "operator:type",
    "operator:short",
    "website",
    "wikidata",
    "internet_access",
    "wheelchair",
    "opening_hours",
    "capacity",
    "historic",
    "heritage",
    "level",
    "building:levels",
    "building:material",
    "building:part",
    "theatre",
    "cinema",
    "studio",
    "music",
    "museum",
    "gallery",
    "artwork_type",
]

# Normalize only the selected tags (if present)
tags_df = pd.json_normalize(gdf["tags"])
tags_df = tags_df[[col for col in tags_df.columns if col in RELEVANT_TAGS]]

# Clean up column names for Pandas
tags_df.columns = (
    tags_df.columns
    .str.replace(":", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# Concatenate with main gdf
gdf = pd.concat(
    [gdf.drop(columns="tags").reset_index(drop=True), tags_df.reset_index(drop=True)],
    axis=1,
)

# Ensure it’s still a GeoDataFrame
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=gdf.crs)


In [44]:
gdf = gdf.rename(
    columns={
        "osm_element_type": "osm_type",
        "osm_element_id": "osm_id",
    }
)

In [45]:
# 1. Project to a metric CRS (UTM zone 17N for Toronto)
gdf_proj = gdf.to_crs(epsg=32617)

# 2. Calculate centroids in projected CRS
gdf_proj["geom_point"] = gdf_proj.geometry.centroid

# 3. Transform centroids back to EPSG:4326
gdf["geom_point"] = gdf_proj["geom_point"].to_crs(epsg=4326)

# Optional: if you want to make the centroid the active geometry
# gdf = gdf.set_geometry("geom_point")


In [46]:
# Keep only the centroid as the geometry
gdf = gdf.drop(columns=["geometry"])

# Rename geom_point to geometry (optional)
gdf = gdf.set_geometry("geom_point")

OUT_PATH = "../../data/venues/eddit_venues_centroids.gpkg"
gdf.to_file(OUT_PATH, layer="venues", driver="GPKG")

SyntaxError: invalid syntax (1664630162.py, line 7)

Compare between Axle and OSM data producing two sets of data which is missing in the other one. We can review this manually